Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo\\SNconsumptionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,6.559,34055.69620,16128.87538,20240.96386,0
2017-01-01 00:10:00,6.414,29814.68354,19375.07599,20131.08434,0
2017-01-01 00:20:00,6.313,29128.10127,19006.68693,19668.43373,0
2017-01-01 00:30:00,6.121,28228.86076,18361.09422,18899.27711,0
2017-01-01 00:40:00,5.921,27335.69620,17872.34043,18442.40964,0


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [5]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (36691, 5)
Las dimensiones de test son:  (10535, 5)
Las dimensiones de val son:  (5190, 5)


Se normalizan los datos

In [6]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [7]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [8]:
datosNormalizados.shape

(52416, 5)

In [9]:
datosNormalizados.head(37)


,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858
2017-01-01 00:50:00,-2.165569,-0.872729,-0.597600,-0.293039,-1.660858
2017-01-01 01:00:00,-2.199865,-0.958984,-0.681090,-0.321667,-1.516340
2017-01-01 01:10:00,-2.223323,-1.035189,-0.746587,-0.396816,-1.516340
2017-01-01 01:20:00,-2.193880,-1.127306,-0.832236,-0.463913,-1.516340


Espacio de búsqueda

In [10]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [11]:
futuros = 24
pasados  = 12

In [12]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 1])


In [13]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52381, 12, 5)
Dimensiones de Y: (52381, 1)


In [14]:
print(datosY[0])

[-1.59375341]


Se dividen nuevamente los conjuntos de datos

In [15]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36666, 12, 5)
Las dimensiones de testX son:  (10529, 12, 5)
Las dimensiones de valX son:  (5186, 12, 5)


In [16]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36666, 1)
Las dimensiones de testY son:  (10529, 1)
Las dimensiones de valY son:  (5186, 1)


Se crean métricas para medir desempeño

In [17]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [18]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [19]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1], trainX.shape[2])))
    if (params['layers'] == 1):
      model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(GRU(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=128,
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [20]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

287/287 - 19s - 65ms/step - ia: 0.2480 - loss: 1.2267 - mae: 0.9102 - rmse: 1.1050 - smape: 1.5068 - val_ia: 0.2173 - val_loss: 0.8542 - val_mae: 0.7688 - val_rmse: 0.9176 - val_smape: 1.9034

Epoch 2/128                                           

287/287 - 9s - 30ms/step - ia: 0.2012 - loss: 1.0900 - mae: 0.8644 - rmse: 1.0427 - smape: 1.5664 - val_ia: 0.2112 - val_loss: 0.8683 - val_mae: 0.7758 - val_rmse: 0.9243 - val_smape: 1.8613

Epoch 3/128                                           

287/287 - 5s - 18ms/step - ia: 0.1762 - loss: 1.0567 - mae: 0.8516 - rmse: 1.0266 - smape: 1.6053 - val_ia: 0.2115 - val_loss: 0.8685 - val_mae: 0.7759 - val_rmse: 0.9243 - val_smape: 1.8414

Epoch 4/128                                           

287/287 - 5s - 18ms/step - ia: 0.1573 - loss: 1.0334 - mae: 0.8434 - rmse: 1.0151 - smape: 1.6402 - val_ia: 0.2129 - val_loss: 0.8641 - val_mae: 0.7738 - val_rmse: 0.9220 - val_smape: 1.8438

Epoch 5

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                        

2292/2292 - 70s - 30ms/step - ia: 0.7611 - loss: 0.2180 - mae: 0.3511 - rmse: 0.4440 - smape: 0.7154 - val_ia: 0.3172 - val_loss: 0.2436 - val_mae: 0.3734 - val_rmse: 0.4146 - val_smape: 0.7522

Epoch 2/128                                                                        

2292/2292 - 83s - 36ms/step - ia: 0.8450 - loss: 0.1006 - mae: 0.2339 - rmse: 0.3072 - smape: 0.5603 - val_ia: 0.3842 - val_loss: 0.1751 - val_mae: 0.3104 - val_rmse: 0.3465 - val_smape: 0.6731

Epoch 3/128                                                                        

2292/2292 - 81s - 36ms/step - ia: 0.8605 - loss: 0.0834 - mae: 0.2120 - rmse: 0.2792 - smape: 0.5186 - val_ia: 0.3805 - val_loss: 0.1990 - val_mae: 0.3296 - val_rmse: 0.3637 - val_smape: 0.6914

Epoch 4/128                                                                        

2292/2292 - 48s - 21ms/step - ia: 0.8721 - loss: 0.0706 - mae: 0.1947 - rms

...

Epoch 1/128                                                                         

2292/2292 - 43s - 19ms/step - ia: 0.3308 - loss: 0.9708 - mae: 0.8133 - rmse: 0.9696 - smape: 1.4055 - val_ia: 0.1860 - val_loss: 0.7861 - val_mae: 0.7326 - val_rmse: 0.7648 - val_smape: 1.2817

Epoch 2/128                                                                         

2292/2292 - 38s - 17ms/step - ia: 0.5658 - loss: 0.5419 - mae: 0.5930 - rmse: 0.7247 - smape: 1.0217 - val_ia: 0.1778 - val_loss: 1.1886 - val_mae: 0.9174 - val_rmse: 0.9506 - val_smape: 1.3568

Epoch 3/128                                                                         

2292/2292 - 41s - 18ms/step - ia: 0.6303 - loss: 0.4346 - mae: 0.5259 - rmse: 0.6491 - smape: 0.9014 - val_ia: 0.1791 - val_loss: 1.1072 - val_mae: 0.8801 - val_rmse: 0.9120 - val_smape: 1.3335

Epoch 4/128                                                                         

2292/2292 - 41s - 18ms/step - ia: 0.6593 - loss: 0.3836 - mae: 0.4918 -

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



4584/4584 - 42s - 9ms/step - ia: 0.2973 - loss: 1.0353 - mae: 0.8350 - rmse: 0.9944 - smape: 1.5374 - val_ia: 0.1392 - val_loss: 0.8240 - val_mae: 0.7543 - val_rmse: 0.7728 - val_smape: 1.7145

Epoch 2/128                                                                          

4584/4584 - 31s - 7ms/step - ia: 0.2989 - loss: 1.0105 - mae: 0.8256 - rmse: 0.9823 - smape: 1.5397 - val_ia: 0.1395 - val_loss: 0.8205 - val_mae: 0.7528 - val_rmse: 0.7713 - val_smape: 1.7084

Epoch 3/128                                                                          

4584/4584 - 27s - 6ms/step - ia: 0.3032 - loss: 0.9942 - mae: 0.8177 - rmse: 0.9740 - smape: 1.5358 - val_ia: 0.1399 - val_loss: 0.8172 - val_mae: 0.7514 - val_rmse: 0.7699 - val_smape: 1.7014

Epoch 4/128                                                                          

4584/4584 - 29s - 6ms/step - ia: 0.3023 - loss: 0.9820 - mae: 0.8132 - rmse: 0.9682 - smape: 1.5386 - val_ia: 0.1403 - val_loss: 0.8137 - val_mae: 0.7498 - v

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                          

287/287 - 7s - 24ms/step - ia: 0.3576 - loss: 0.8578 - mae: 0.7633 - rmse: 0.9222 - smape: 1.3551 - val_ia: 0.4114 - val_loss: 0.8105 - val_mae: 0.7560 - val_rmse: 0.8837 - val_smape: 1.3043

Epoch 2/128                                                                          

287/287 - 3s - 12ms/step - ia: 0.5644 - loss: 0.5616 - mae: 0.6024 - rmse: 0.7478 - smape: 1.0416 - val_ia: 0.4385 - val_loss: 1.1194 - val_mae: 0.8895 - val_rmse: 1.0273 - val_smape: 1.2996

Epoch 3/128                                                                          

287/287 - 3s - 12ms/step - ia: 0.6200 - loss: 0.4867 - mae: 0.5549 - rmse: 0.6962 - smape: 0.9485 - val_ia: 0.4441 - val_loss: 1.2310 - val_mae: 0.9266 - val_rmse: 1.0710 - val_smape: 1.2884

Epoch 4/128                                                                          

287/287 - 3s - 12ms/step - ia: 0.6524 - loss: 0.4306 - mae: 0.5202 - rmse: 0

In [21]:
print(best)

{'activation': 1, 'batch': 1, 'dropout': 0.4, 'layers': 2.0, 'learning_rate': 8.360431152477965e-05, 'units': 4}
